# 🚀 TabDrift: Native Drifting Models for Tabular Data Generation (GPU Execution)

This notebook runs the complete end-to-end TabDrift pipeline on GPU (Kaggle).

- **Model**: Residual MLP TabDrift Generator (~10.6M parameters)
- **Sampling**: Native 1-Step Direct Pass ($K=1$ NFE)
- **Execution Time**: ~15 minutes for 2000 epochs on Tesla T4 GPU!

In [ ]:
# 1. Clone Private GitHub Repository using Kaggle User Secrets
import os
from kaggle_secrets import UserSecretsClient

# Fetch secret token named GITHUB_TOKEN from Kaggle Add-ons -> Secrets
user_secrets = UserSecretsClient()
github_token = user_secrets.get_secret("GITHUB_TOKEN")

# Set your GitHub username and repo name below
GITHUB_USER = "ahmed-fouad-lagha"
REPO_NAME = "tabsyn"

# Clone repository and change working directory
!git clone https://{github_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git
%cd {REPO_NAME}

In [ ]:
# 2. Install Required Dependencies for TabDrift
!pip install -q icecream zero tomli tomli-w category_encoders executing asttokens prdc

In [ ]:
# 3. Verify GPU Availability
!nvidia-smi

In [ ]:
# 4. Train TabDrift Generator for 2000 Epochs (Drift Scale c=1.5, Full LR decay)
!PYTHONPATH=. python tabsyn/drift_train.py \
    --dataname adult \
    --gpu 0 \
    --epochs 2000 \
    --batch_size 4096 \
    --lr 1e-4 \
    --temperatures 0.1 0.5 1.0 2.0 \
    --drift_scale 1.5 \
    --patience 2000

In [ ]:
# 5. Generate 32,561 Synthetic Rows (1-NFE Direct Pass)
!PYTHONPATH=. python tabsyn/drift_sample.py \
    --dataname adult \
    --gpu 0 \
    --steps 1

In [ ]:
# 6. Full Multi-Classifier Machine Learning Efficacy Evaluation (XGBoost, RandomForest, MLP, LogisticRegression)
!python eval/eval_mle.py --dataname adult --model tabdrift